## Running from Collab:


In [2]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
172 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as r

In [3]:
import subprocess
import time
# Start the server as a background process
process = subprocess.Popen("ollama serve", shell=True)

# Wait a few seconds for the server to initialize
time.sleep(5)

In [4]:
!ollama pull llama3.2

In [7]:
!pip install chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

In [9]:
#import os
import chromadb
#import requests
from openai import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
#from PyPDF2 import PdfReader
#from google.colab import userdata
#from google import genai
from sentence_transformers import CrossEncoder

KeyboardInterrupt: 

# Knowledge Base:

### Loading previous knowledge base:
-if loading previous knowledge base, just run the following cell   
-if building knowledge base, run all cells in this section

In [ ]:
# chroma_client = chromadb.PersistentClient(path='/content/')
# collection = client.create_collection(
#     name="my_collection",
#     embedding_function=OpenAIEmbeddingFunction(
#         model_name="text-embedding-3-small"
#         api_key_env_var=OPENAI_API_KEY
#     )
# )

chroma_client = chromadb.PersistentClient(path='./chroma_db')
collection = chroma_client.get_or_create_collection(name="test_collection")

In [ ]:
def get_text_txt_md(file_path: str) -> str:
  with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()
  return text

def get_text_pdf(pdf_path: str) -> str:
    """Extract raw text from a PDF file."""
    try:
        reader = PdfReader(pdf_path)
        return " ".join(page.extract_text() for page in reader.pages if page.extract_text())
    except Exception as e:
        raise RuntimeError(f"Error reading PDF: {e}")

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 25) -> list[str]:
    """Split text into manageable chunks for embeddings."""
    words = text.split()
    return [" ".join(words[i-overlap:i+chunk_size]) for i in range(overlap, len(words), chunk_size - overlap)]

In [ ]:
files_paths = ['toy_rag_data/Snake_wikiWikipedia.pdf', 'toy_rag_data/cool_math.pdf', 'toy_rag_data/its_nice_that.txt']
chunks = []
text = ""
for file in files_paths:
  if "pdf" in file:
    text = get_text_pdf(file)
  else:
    text = get_text_txt_md(file)
  chunks += chunk_text(text)

collection.upsert(
    documents=chunks,
    ids=[f"id{i}" for i in range(len(chunks))]
)

In [ ]:
#test retrieve:
collection.query(
      query_texts=["Who was the original creator?"],
      n_results=4
  )

# Generation:

In [ ]:
rr_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
def rerank(chunks, prompt, k, rr_model){
    return rr_model.rank(query, chunks, return_documents=True, top_k=3)
}

In [ ]:
def call_llm(name: str, model: str, prompt: str, client):
  if name == "gemini":
    try:
      response = client.interactions.create(
          model=model,
          input=prompt
      )
      return response.output_text
    except Exception as e:
        raise RuntimeError(f"Gemini LLM query failed: {e}")
  if name == "ollama":
    try:
      response = client.chat.completions.create(
          model=model,
          messages=[
              {"role": "user", "content": prompt}
          ]
      )
      return response.choices[0].message.content
    except Exception as e:
        raise RuntimeError(f"Ollama LLM query failed: {e}")

In [ ]:
def gen_response(query,  client, rr_model, collection, k, kp):
  k_chunks = collection.query(
      query_texts=[query],
      n_results=k
  )
  kp_chunks = rr_model.rank(query, chunks, return_documents=True, top_k=kp);
  kp_chunks = [item["text"] for item in kp_chunks]
  context = "\n".join(kp_chunks)
  prompt = f"{query} Only use the following context to answer this question. Clearly state when the context does not contain the answer: {context}"
  return call_llm("ollama", "gpt-oss:20b", prompt, client), k_chunks, kp_chunks


In [ ]:
client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)
rr_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
query = "When was the snake game made?"
response, k_chunks, kp_chunks = gen_response(query, client, rr_model, collection, 10, 3)
print(response)
print(kp_chunks)

In [ ]:
from openai import OpenAI
import ollama
ollama.pull('llama3.2')

client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)

responses_result = client.responses.create(
  model='llama3.2',
  input='repeat this sentence: this is a test',
)
print(responses_result.output_text)